# RadioML 2016.10A

This notebook shows how to download and use the RadioML 2016.10A dataset

### Data Gathering Pipeline

Kaggle -> .zip file -> pickle file -> python export script

## 1. Kaggle -> pkl(Pickle) file 

Go to this link: https://www.kaggle.com/datasets/nolasthitnotomorrow/radioml2016-deepsigcom/data

    a. Download the dataset using any method and obtain the .zip file
    
    b. Unzip the file to get the pkl file

## 2. Unpickle File

After running the script, the file will be saved as an npz file for easier use and less space. You can delete the pkl file if you would like.

In [ ]:
%pip install kagglehub
%pip install numpy

In [ ]:
import pickle
import numpy as np

def convert_label(label):
    label_map = {
        0: "BPSK",
        1: "QPSK",
        2: "8PSK",
        3: "PAM4",
        4: "QAM16",
        5: "QAM64",
        6: "CPFSK",
        7: "GFSK",
        8: "AM-DSB",
        9: "AM-SSB",
        10: "WBFM"
    }
    return label_map.get(label, "Unknown")

# Load the dataset
with open("RML2016.10a_dict.pkl", "rb") as f:
    data = pickle.load(f, encoding="latin1")

# data is dict: (mod_type, snr) -> array of shape [1000, 2, 128]
X = []
labels = []
snrs = []
mod_types = sorted(list(set([k[0] for k in data.keys()])))
for (mod, snr), samples in data.items():
    X.append(samples)
    labels += [mod_types.index(mod)] * samples.shape[0]
    snrs += [snr] * samples.shape[0]

X = np.vstack(X)  # shape: [220000, 2, 128]
y = np.array(labels)

np.savez('radioml_2016.10a.npz', X=X, y=y, snrs=np.array(snrs))

## Example of how to Load Data

Note: The convert_label() function shows the mapping from number value to name of type

In [ ]:
data = np.load('radioml_2016.10a.npz')
X = data['X']
y = data['y']
snrs = data['snrs']
example_data = X[0]

print("Dataset shape:", X.shape)
print(convert_label(y[0]))
print("Number of classes:", len(mod_types))
print("Example SNRs:", sorted(set(snrs)))
print("Example data sample:", example_data)
print("Example data shape:", example_data.shape)

In [ ]:
def hybrid_preprocess(X_train, window='hamming', log_normalize='global'):
    """
    Build 3-channel hybrid input: [I, Q, log_power_spectrum].
    X_train, X_test: (N, 2, 128). Returns (N, 3, 128) float32 each.
    log_normalize: 'global' = use train mean/std for both (recommended); None to skip.
    """
    def _log_power(X, win):
        X = np.asarray(X, dtype=np.float64)
        N, _, L = X.shape
        iq = X[:, 0, :] + 1j * X[:, 1, :]
        iq = iq * win
        spec = np.fft.fft(iq, axis=-1)
        power = (spec.real ** 2 + spec.imag ** 2).astype(np.float32)
        return np.log1p(power)

    L = X_train.shape[-1]
    win = np.hanning(L) if window == 'hann' else (np.hamming(L) if window == 'hamming' else np.ones(L))

    log_power_train = _log_power(X_train, win)

    if log_normalize == 'global':
        mean, std = log_power_train.mean(), log_power_train.std()
        if std > 0:
            log_power_train = (log_power_train - mean) / std

    # Channels: I, Q, log_power
    I_train = np.asarray(X_train[:, 0, :], dtype=np.float32)
    Q_train = np.asarray(X_train[:, 1, :], dtype=np.float32)

    X_hybrid_train = np.stack([I_train, Q_train, log_power_train], axis=1)   # (N, 3, 128)
    return X_hybrid_train


In [ ]:
output = hybrid_preprocess(X)

In [ ]:
print( output.shape )
print( output.dtype )

In [ ]:
import socket as sock
import struct

In [ ]:
def recv_exact(sock, n):
    buf = bytearray(n)
    view = memoryview(buf)
    while n:
        received = sock.recv_into(view, n)
        if received == 0: raise ConnectionError("Socket closed")
        view = view[received:]
        n -= received
    return buf

In [103]:
batch_len = output.shape[0]
batch = np.ascontiguousarray(output)
print(batch.shape)
print(batch[0].nbytes)

(220000, 3, 128)
1536


In [ ]:
from tqdm import tqdm

In [104]:
HOST = "127.0.0.1"
PORT = 8080
BUNDLE_COUNT = 1000

with sock.create_connection( (HOST, PORT) ) as s:
    s.sendall(struct.pack("!I", batch_len))

    n_correct = 0

    bar = tqdm( range( 0, batch_len, BUNDLE_COUNT ) )

    for i in bar:
        s.sendall(batch[i:i+BUNDLE_COUNT].tobytes())

        data = recv_exact(s, BUNDLE_COUNT)
        res = np.frombuffer( data, dtype=np.uint8 )

        valid = (y[i:i+BUNDLE_COUNT] == res)
        n_correct += np.sum(valid)

        bar.set_postfix_str(f"Recv'd bundle {i} to {i + BUNDLE_COUNT}, got {np.sum(valid)} / {BUNDLE_COUNT} correct, accuracy: {n_correct / (i + BUNDLE_COUNT)}")

    print( f" Got {n_correct} / 220000, accuracy: {n_correct / 220000} " )


100%|██████████| 220/220 [04:20<00:00,  1.19s/it, Recv'd bundle 219000 to 220000, got 1 / 1000 correct, accuracy: 0.5594954545454546]   

 Got 123089 / 220000, accuracy: 0.5594954545454546 
